# QT001: Core BB84 Protocol Design and Multi-Vector Threat Modeling

This notebook implements the foundational BB84 simulation engine with channel noise modeling and an adaptive Intercept-Resend (IR) threat model.

**GitHub:** https://github.com/sreecharan-desu/bb84-qkd

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

class BB84Simulation:
    def __init__(self, n_bits: int = 2000):
        self.n_bits = n_bits
        # Alice's secret bits and randomly chosen bases
        self.alice_bits = np.random.randint(2, size=n_bits)
        self.alice_bases = np.random.randint(2, size=n_bits)  # 0: Plus (+), 1: Cross (x)

    def run_protocol(self, p_noise: float = 0.03, eve_intensity: float = 0.0) -> np.ndarray:
        """
        Simulates one full transmission session.
        eve_intensity (0.0 to 1.0): fraction of qubits Eve intercepts.
        Returns bob_results array (-1 means sifted out, rest are measured values).
        """
        bob_bases = np.random.randint(2, size=self.n_bits)
        bob_results = np.zeros(self.n_bits, dtype=int)

        for i in range(self.n_bits):
            # Eve intercepts with probability eve_intensity
            is_intercepted = random.random() < eve_intensity

            if self.alice_bases[i] == bob_bases[i]:
                # When Eve guesses the wrong basis (50% chance), she introduces an error.
                # This is the 25% error contribution from a full IR attack.
                induced_err = 0.25 if is_intercepted else 0.0
                total_p = p_noise + induced_err
                bob_results[i] = 1 - self.alice_bits[i] if (random.random() < total_p) else self.alice_bits[i]
            else:
                bob_results[i] = -1  # Basis mismatch — sifted out

        return bob_results

def calculate_qber(alice_bits, bob_results):
    mask = bob_results != -1
    if not any(mask): return 0.0
    errors = alice_bits[mask] != bob_results[mask]
    return np.mean(errors)

# Run demonstration
sim = BB84Simulation(10000)
qber_clean = calculate_qber(sim.alice_bits, sim.run_protocol(p_noise=0.04, eve_intensity=0.0))
qber_attack = calculate_qber(sim.alice_bits, sim.run_protocol(p_noise=0.04, eve_intensity=0.15))  # 15% Eve

print(f"QBER (Clean Channel, 4% noise)     : {qber_clean*100:.2f}%")
print(f"QBER (15% Eve + 4% noise)          : {qber_attack*100:.2f}%")
print(f"")
print(f"Key observation: A 15% Eve activity blends into a 4% noisy channel.")
print(f"The total QBER ({qber_attack*100:.2f}%) may still be below the 11% abort threshold!")
print(f"This is the 'Sub-Threshold Gap' that QT002 and QT003 are designed to close.")